In [63]:
import pandas as pd

In [64]:
a = pd.read_csv('../data/mexico-real-estate-1.csv')
b = pd.read_csv('../data/mexico-real-estate-2.csv')
c = pd.read_csv('../data/mexico-real-estate-3.csv')

In [65]:
display(a.info())
display(b.info())
display(c.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  700 non-null    object 
 1   state          700 non-null    object 
 2   lat            583 non-null    float64
 3   lon            583 non-null    float64
 4   area_m2        700 non-null    float64
 5   price_usd      700 non-null    object 
dtypes: float64(3), object(3)
memory usage: 32.9+ KB


None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  700 non-null    object 
 1   state          700 non-null    object 
 2   lat            571 non-null    float64
 3   lon            571 non-null    float64
 4   area_m2        700 non-null    float64
 5   price_mxn      700 non-null    float64
dtypes: float64(4), object(2)
memory usage: 32.9+ KB


None

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   property_type            700 non-null    object 
 1   place_with_parent_names  700 non-null    object 
 2   lat-lon                  582 non-null    object 
 3   area_m2                  700 non-null    float64
 4   price_usd                700 non-null    float64
dtypes: float64(2), object(3)
memory usage: 27.5+ KB


None

In [66]:
display(a[["lat", "lon"]].isna().mean() * 100)
display(b[["lat", "lon"]].isna().mean() * 100)
display(c[["lat-lon"]].isna().mean() * 100)

lat   16.71
lon   16.71
dtype: float64

lat   18.43
lon   18.43
dtype: float64

lat-lon   16.86
dtype: float64

In [67]:
a.head(3)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.56,-99.23,150.00,"$67,965.56"
1,house,Nuevo León,25.69,-100.20,186.00,"$63,223.78"
2,apartment,Guerrero,16.77,-99.76,82.00,"$84,298.37"


In [68]:
b.head(3)

,property_type,state,lat,lon,area_m2,price_mxn
0,apartment,Nuevo León,25.72,-100.35,72.00,"1,300,000.00"
1,apartment,Puebla,NaN,NaN,190.00,"2,500,000.00"
2,house,Morelos,23.63,-102.55,360.00,"5,300,000.00"


In [69]:
c.head(3)

,property_type,place_with_parent_names,lat-lon,area_m2,price_usd
0,apartment,|México|Distrito Federal|Gustavo A. Madero|Acu...,"19.52589,-99.151703",71.00,"48,550.59"
1,house,|México|Estado de México|Toluca|Metepec|,"19.2640539,-99.5727534",233.00,"168,636.73"
2,house,|México|Estado de México|Toluca|Toluca de Lerd...,"19.268629,-99.671722",300.00,"86,932.69"


#### Handling missing lat and lon data
The next granule of geospatial data is state, which is too large to impute lat-lon coordinates even if grouped by state. Since missing values make up only about 18% of each dataframe, it is better to drop these rows and keep over 80% of real, accurate data.

In [70]:
a = a.dropna(subset=['lat', 'lon']).assign(
    price_usd = lambda x: x['price_usd']
    .str.replace('$', '')
    .str.replace(',', '')
)

a.head(3)

,property_type,state,lat,lon,area_m2,price_usd
0,house,Estado de México,19.56,-99.23,150.00,67965.56
1,house,Nuevo León,25.69,-100.20,186.00,63223.78
2,apartment,Guerrero,16.77,-99.76,82.00,84298.37


In [71]:
b = b.dropna(subset=['lat', 'lon']).assign(
    price_usd = lambda x: x['price_mxn'] / 19
).drop('price_mxn', axis=1)

b.head(3)

,property_type,state,lat,lon,area_m2,price_usd
0,apartment,Nuevo León,25.72,-100.35,72.00,"68,421.05"
2,house,Morelos,23.63,-102.55,360.00,"278,947.37"
6,apartment,Estado de México,19.27,-99.57,85.00,"65,789.47"


In [72]:
c = (c.dropna(subset=['lat-lon']).assign(
    lat = lambda x: x['lat-lon'].str.split(',', expand=True)[0],
    lon = lambda x: x['lat-lon'].str.split(',', expand=True)[1],
    state = lambda x: x['place_with_parent_names'].str.split('|', expand=True)[2]
).drop(['place_with_parent_names', 'lat-lon'], axis=1)
.reindex(columns=['property_type', 'state', 'lat', 'lon', 'area_m2', 'price_usd']))

c.head(3)

,property_type,state,lat,lon,area_m2,price_usd
0,apartment,Distrito Federal,19.52589,-99.151703,71.00,"48,550.59"
1,house,Estado de México,19.2640539,-99.5727534,233.00,"168,636.73"
2,house,Estado de México,19.268629,-99.671722,300.00,"86,932.69"


In [73]:
df = pd.concat([a, b, c]).reset_index(drop=True)
df.shape

(1736, 6)

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1736 entries, 0 to 1735
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  1736 non-null   object 
 1   state          1736 non-null   object 
 2   lat            1736 non-null   object 
 3   lon            1736 non-null   object 
 4   area_m2        1736 non-null   float64
 5   price_usd      1736 non-null   object 
dtypes: float64(1), object(5)
memory usage: 81.5+ KB


In [75]:
df['price_usd'] = df['price_usd'].astype('float')
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
df.describe(include='all')

,property_type,state,lat,lon,area_m2,price_usd
count,1736,1736,"1,736.00","1,736.00","1,736.00","1,736.00"
unique,2,30,"1,502.00","1,465.00",NaN,NaN
top,house,Distrito Federal,23.63,-102.55,NaN,NaN
freq,1329,303,30.00,29.00,NaN,NaN
mean,NaN,NaN,NaN,NaN,170.26,"115,331.98"
std,NaN,NaN,NaN,NaN,80.59,"65,426.17"
min,NaN,NaN,NaN,NaN,60.00,"33,157.89"
25%,NaN,NaN,NaN,NaN,101.75,"65,789.47"
50%,NaN,NaN,NaN,NaN,156.00,"99,262.13"
75%,NaN,NaN,NaN,NaN,220.00,"150,846.66"


In [76]:
df['property_type'].value_counts()

property_type
house        1329
apartment     407
Name: count, dtype: int64

In [77]:
df['state'].value_counts()

state
Distrito Federal                   303
Estado de México                   179
Yucatán                            171
Morelos                            160
Querétaro                          128
Veracruz de Ignacio de la Llave    117
Puebla                              95
Nuevo León                          83
Jalisco                             60
San Luis Potosí                     55
Chiapas                             55
Guerrero                            49
Tamaulipas                          48
Quintana Roo                        38
Baja California                     29
Sinaloa                             26
Chihuahua                           20
Tabasco                             20
Hidalgo                             17
Baja California Sur                 15
Sonora                              12
Guanajuato                          12
Aguascalientes                      10
Nayarit                              9
Durango                              7
Tlaxcala           

In [78]:
df.to_csv('../data/mexico-real-estate.csv', index=False)